# Data Converter — `.npy` to YOLO Format

Converts annotation `.npy` label files to YOLO `.txt` format and writes `configs/dataset.yaml`.  
**Run this once before any training notebook.**

In [ ]:
import subprocess, sys

for pkg in ['pyyaml', 'matplotlib']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

In [ ]:
from pathlib import Path
from collections import defaultdict
import numpy as np
import yaml
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

In [ ]:
ROOT        = Path('../../../').resolve()
DATASET_DIR = ROOT / 'data/organisms_data'
CONFIGS_DIR = ROOT / 'models/organism_det/configs'
PLOTS_DIR   = ROOT / 'models/organism_det/results/plots'

SPLITS       = ['train', 'val', 'test']
IMG_W, IMG_H = 1280, 720

CLASS_MAP = {
    'Trichomonas vaginalis':           0,
    'Bacterial vaginosis flora shift': 1,
    'Candida spp.':                    2,
    'Actinomyces spp.':                3,
}
SKIP_ORGANISMS = {'None', 'Unknown', 'Reactive changes'}
CLASS_NAMES    = {v: k for k, v in CLASS_MAP.items()}
CLASS_COLORS   = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12']

In [ ]:
def convert_npy_to_yolo(npy_path: Path, out_path: Path) -> int:
    data = np.load(npy_path, allow_pickle=True).item()
    lines = []
    for bbox, attr in zip(data['bboxes'], data['attributes']):
        organism = attr.get('Organisms', 'Unknown')
        if organism not in CLASS_MAP:
            continue
        x1, y1, x2, y2 = bbox
        cx = (x1 + x2) / 2 / IMG_W
        cy = (y1 + y2) / 2 / IMG_H
        w  = (x2 - x1) / IMG_W
        h  = (y2 - y1) / IMG_H
        lines.append(f'{CLASS_MAP[organism]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    out_path.write_text('\n'.join(lines))
    return len(lines)

In [ ]:
for split in SPLITS:
    label_dir = DATASET_DIR / split / 'labels'
    n_files, n_instances, n_empty = 0, 0, 0
    for npy_file in sorted(label_dir.glob('*_seg.npy')):
        stem     = npy_file.stem.replace('_seg', '')
        out_path = label_dir / f'{stem}.txt'
        n        = convert_npy_to_yolo(npy_file, out_path)
        n_files     += 1
        n_instances += n
        n_empty     += int(n == 0)
    print(f'[{split:5s}]  {n_files} files  |  {n_instances:4d} instances  |  {n_empty} empty label files')

### Verify class distribution across splits

In [ ]:
counts = defaultdict(lambda: defaultdict(int))
for split in SPLITS:
    label_dir = DATASET_DIR / split / 'labels'
    for txt in label_dir.glob('*.txt'):
        for line in txt.read_text().strip().splitlines():
            if line:
                counts[split][int(line.split()[0])] += 1

header = f"{'Class':<42} {'train':>6} {'val':>6} {'test':>6}  {'total':>6}"
print(header)
print('-' * len(header))
for cls_id in range(4):
    row = [counts[s][cls_id] for s in SPLITS]
    print(f'{CLASS_NAMES[cls_id]:<42} {row[0]:>6} {row[1]:>6} {row[2]:>6}  {sum(row):>6}')

### Write `configs/dataset.yaml`

In [ ]:
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)

dataset_cfg = {
    'path':  str(DATASET_DIR),
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc':    len(CLASS_MAP),
    'names': list(CLASS_MAP.keys()),
}

yaml_path = CONFIGS_DIR / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_cfg, f, default_flow_style=False, sort_keys=False)

print(f'Written: {yaml_path}')
print(yaml_path.read_text())

### Visual sanity check — draw converted labels on sample images

In [ ]:
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

img_dir   = DATASET_DIR / 'train' / 'images'
label_dir = DATASET_DIR / 'train' / 'labels'
samples   = sorted(img_dir.glob('*.jpg'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(24, 10))
for ax, img_path in zip(axes.flat, samples):
    txt_path = label_dir / f'{img_path.stem}.txt'
    ax.imshow(Image.open(img_path))
    if txt_path.exists():
        for line in txt_path.read_text().strip().splitlines():
            if not line:
                continue
            cls, cx, cy, w, h = map(float, line.split())
            cls = int(cls)
            x1  = (cx - w / 2) * IMG_W
            y1  = (cy - h / 2) * IMG_H
            rect = patches.Rectangle(
                (x1, y1), w * IMG_W, h * IMG_H,
                linewidth=2, edgecolor=CLASS_COLORS[cls], facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, CLASS_NAMES[cls],
                    color=CLASS_COLORS[cls], fontsize=7, fontweight='bold')
    ax.axis('off')
    ax.set_title(img_path.stem, fontsize=9)

fig.suptitle('Label Sanity Check — Train Samples', fontsize=14, y=1.01)
plt.tight_layout()
out_path = PLOTS_DIR / 'label_sanity_check.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')